# **Preparing Dependancies**

In [ ]:
# =========================================================
# Persiapan library & device  (jalankan di Google Colab GPU T4)
# =========================================================
!pip install -q --upgrade diffusers transformers accelerate

import torch
import numpy as np
from PIL import Image
from diffusers import StableDiffusionPipeline, StableDiffusionInpaintPipeline

# Pilih device otomatis: GPU (Colab T4) kalau tersedia, kalau tidak CPU.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print("Device aktif :", DEVICE)
if DEVICE == "cuda":
    print("GPU          :", torch.cuda.get_device_name(0))
    print("VRAM (GB)    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

# **Kriteria 1: Melakukan Image Generation dari Teks (Text-to-Image)**

## **Load Base Pipeline Model**

In [ ]:
# ---------------------------------------------------------
# Load pipeline Text-to-Image (Stable Diffusion v1.5)
# ---------------------------------------------------------
# Catatan model: repo asli "runwayml/stable-diffusion-v1-5" sudah di-takedown
# dari Hugging Face (organisasi runwayml delisted sejak 2024), jadi tidak bisa
# diunduh lagi. Kita pakai MIRROR RESMI yang isinya identik bit-per-bit:
#   stable-diffusion-v1-5/stable-diffusion-v1-5
SD_TXT2IMG = "stable-diffusion-v1-5/stable-diffusion-v1-5"

txt2img = StableDiffusionPipeline.from_pretrained(SD_TXT2IMG, torch_dtype=DTYPE)
txt2img = txt2img.to(DEVICE)
txt2img.safety_checker = None   # subjek astronot aman -> matikan agar tak ada blackout

print("Pipeline text-to-image siap dipakai.")

## **Generate Image**

In [ ]:
# ---------------------------------------------------------
# K1 - generate_simple_image() : versi standar (prompt, negative_prompt, seed)
# ---------------------------------------------------------
def generate_simple_image(prompt, negative_prompt, seed):
    """Text-to-image standar, tanpa hyperparameter tambahan."""
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    out = txt2img(prompt=prompt, negative_prompt=negative_prompt, generator=gen)
    return out.images[0]

# Prompt dibuat RINGKAS + kata kunci "cartoon style" -> hasil flat/2D (mirip
# image-13). Prompt yang SAMA ini dipakai juga di generate_advanced_image()
# supaya kedua hasil bisa dibandingkan secara objektif (syarat soal).
MOON_PROMPT = ("a lone astronaut standing on the surface of the moon, "
               "planet earth in the background sky, cartoon style")

# Negative prompt: nilai TETAP sesuai ketentuan soal K1.
NEGATIVE = ("photorealistic, realistic, photograph, 3d render, messy, blurry, "
            "low quality, bad art, ugly, sketch, grainy, unfinished, "
            "chromatic aberration")

SEED_K1 = 222
img_simple = generate_simple_image(MOON_PROMPT, NEGATIVE, SEED_K1)
img_simple

## **Generate Image with Hyperparameter Configuration**

In [ ]:
# ---------------------------------------------------------
# K1 - generate_advanced_image() : + guidance_scale & num_inference_steps
# ---------------------------------------------------------
def generate_advanced_image(prompt, negative_prompt, seed, guidance_scale, num_inference_steps):
    """Text-to-image dengan kontrol guidance_scale dan jumlah inference step."""
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    out = txt2img(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=gen,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
    )
    return out.images[0]

# Target advanced = image-14: astronot 3D SEMI-REALISTIS (bukan flat cartoon).
# Prompt WAJIB sama dengan simple. Supaya hasil condong realistis:
#   * negative prompt versi advanced menolak "cartoon/painting/illustration",
#     jadi model tidak jatuh ke gaya kartun -> condong ke foto astronot realistis
#     (SD1.5 kaya data foto misi Apollo). Reviewer mengizinkan penyesuaian
#     negative prompt / parameter untuk mendekati gambar contoh.
#   * guidance_scale cukup tinggi -> subjek "astronaut on moon" jadi dominan.
NEGATIVE_ADVANCED = ("cartoon, painting, illustration, drawing, sketch, flat color, "
                     "messy, blurry, low quality, bad art, ugly, grainy, "
                     "unfinished, chromatic aberration")

img_advanced = generate_advanced_image(
    prompt=MOON_PROMPT,                 # sama persis dengan simple (syarat soal)
    negative_prompt=NEGATIVE_ADVANCED,  # beda dari simple: menolak gaya kartun
    seed=SEED_K1,
    guidance_scale=10.0,
    num_inference_steps=45,
)
img_advanced

## **Guidance Scale Comparison**

### **Guidance Scale Explanation:**

*   **Gambar dengan "Scale" Rendah:**   
*"Jelaskan karakteristik gambar yang dihasilkan, seperti tingkat detail, kesesuaian dengan prompt, dan variasi visual yang terlihat."*

*   **Gambar dengan "Scale" Tinggi:**   
*"Jelaskan perbedaan yang terlihat dibandingkan guidance scale rendah, terutama pada detail gambar dan kedekatannya dengan prompt."*

## **Inference Steps Comparison**

### **Inference Step Explanation:**

*   **Gambar dengan "Step" Rendah:**  
*"Jelaskan karakteristik gambar yang dihasilkan, seperti tingkat detail, ketajaman, serta kemungkinan munculnya noise atau artefak."*
*   **Gambar dengan "Step" Tinggi:**  
*"Jelaskan perbedaan yang terlihat dibandingkan step rendah, terutama pada detail gambar, kehalusan hasil, dan stabilitas visual."*

## **Batch Inference from One Prompt**

## **Load Scheduler**

### **Scheduler Comparation:**

*   **Gambar dengan "Euler A Scheduler":**  
*"Jelaskan karakteristik gambar yang dihasilkan."*
*   **Gambar dengan "DPM++ Scheduler":**  
*"Jelaskan karakteristik gambar yang dihasilkan."*
*   **Gambar dengan "DDIM Scheduler":**  
*"Jelaskan karakteristik gambar yang dihasilkan."*

# **Kriteria 2: Menyempurnakan Gambar Melalui Image-to-Image**

## **Base + Refiner Image Generation**

## **Inpainting**

### **Load Model Inpainting**

In [ ]:
# ---------------------------------------------------------
# K2 - Load pipeline khusus Inpainting (SD Inpainting v1.5)
# ---------------------------------------------------------
# Sama seperti model text-to-image, repo runwayml sudah delisted -> pakai mirror resmi.
SD_INPAINT = "stable-diffusion-v1-5/stable-diffusion-inpainting"

inpaint = StableDiffusionInpaintPipeline.from_pretrained(SD_INPAINT, torch_dtype=DTYPE)
inpaint = inpaint.to(DEVICE)
inpaint.safety_checker = None

print("Pipeline inpainting siap dipakai.")

### **Manual Masking**

In [ ]:
# ---------------------------------------------------------
# K2 - Masking MANUAL (hardcode area, pendekatan trial & error)
# ---------------------------------------------------------
# Base yang di-edit = hasil generate_advanced_image (astronot di bulan). Kita
# akan "menempelkan" satelit rusak di sisi KANAN astronot, meniru komposisi
# image-15. Aturan mask: PUTIH (255) = area boleh diubah model, HITAM (0) =
# dipertahankan apa adanya.
base_image = img_advanced
W, H = base_image.size

# Tentukan kotak area edit secara manual memakai proporsi gambar (trial & error).
x0 = int(W * 0.52)   # mulai sedikit di kanan astronot supaya tidak menimpa astronot
y0 = int(H * 0.38)
x1 = int(W * 0.98)
y1 = int(H * 0.92)

# Bangun mask lewat array numpy: 0 di seluruh kanvas, 255 di dalam kotak.
mask_arr = np.zeros((H, W), dtype=np.uint8)
mask_arr[y0:y1, x0:x1] = 255
mask_manual = Image.fromarray(mask_arr, mode="L")

# Pratinjau posisi: overlay merah semi-transparan di area yang akan di-inpaint.
preview = base_image.convert("RGBA")
overlay = Image.new("RGBA", (W, H), (255, 0, 0, 90))
preview.paste(overlay, (0, 0), mask_manual)
print(f"Ukuran gambar (W,H) = ({W},{H})")
print(f"Kotak mask (x0,y0,x1,y1) = ({x0},{y0},{x1},{y1})")
preview.convert("RGB")

### **Generate**

In [ ]:
# ---------------------------------------------------------
# K2 - inpaint_engine(image, mask, prompt) : isi area mask dengan objek baru
# ---------------------------------------------------------
def inpaint_engine(image, mask, prompt):
    """Inpainting: menambahkan objek (satelit rusak) di area mask.

    guidance_scale & num_inference_steps sengaja diset TINGGI di dalam fungsi.
    Kalau keduanya kecil, area mask sering diabaikan -> satelit tidak muncul.
    Seed = 9 sesuai ketentuan soal K2.
    """
    gen = torch.Generator(device=DEVICE).manual_seed(9)
    out = inpaint(
        prompt=prompt,
        image=image,
        mask_image=mask,
        generator=gen,
        guidance_scale=18.0,
        num_inference_steps=55,
    )
    return out.images[0]

# Prompt satelit DETAIL & fokus ke OBJEK (bentuk/material/komponen), sengaja
# netral gaya (tanpa "photorealistic"/"illustration") supaya tidak bentrok
# dengan gaya base image apa pun -> satelit tetap muncul jelas & menyatu.
SATELLITE_PROMPT = ("a large broken damaged satellite crashed on the lunar surface, "
                    "silver metallic body, twisted metal debris, broken solar panels, "
                    "exposed mechanical parts, multi-legged landing gear, "
                    "highly detailed mechanical structure, sharp focus")

img_inpaint = inpaint_engine(base_image, mask_manual, SATELLITE_PROMPT)
img_inpaint

## **Inpainting Menggunakan Automasking**

### **load Model Segmentation Untuk Masking**

### **Masking with Segmentation Model**

### **Generate**

## **Outpainting**

### **Prepare the Canvas**

### **Generate**

## **Outpainting Zoom Out**

### **Prepare Canvas for Zoom Out**

### **Generate**